# 11_a Direct Video Download Shard 1/3

この notebook は `notebooks/11_download_statcast_and_video_sources.ipynb` で `manifests/video_sources_v1.parquet` を作った後に使います。

目的は direct `media_url` を持つ動画だけを shard 1/3 としてダウンロードすることです。`11_a`, `11_b`, `11_c` は別々の Colab runtime / tab で同時に実行できます。同時書き込み衝突を避けるため、この notebook は canonical `download_manifest_v1.parquet` ではなく shard 専用 manifest に書きます。3つが終わったら `11_d_merge_video_download_shards.ipynb` を実行してください。

In [ ]:
from pathlib import Path
import os
import sys
import importlib.util


def _mount_drive_if_needed() -> None:
    try:
        from google.colab import drive  # type: ignore
    except ModuleNotFoundError:
        print('Google Colab ではないため Drive mount を skip します。')
        return

    mountpoint = Path('/content/drive')
    mountpoint.mkdir(parents=True, exist_ok=True)
    if (mountpoint / 'MyDrive').exists():
        print('Drive already mounted at /content/drive')
        return
    try:
        drive.mount(str(mountpoint))
    except ValueError as exc:
        message = str(exc)
        if 'Mountpoint must not already contain files' in message or 'already mounted' in message:
            print(f'Drive mount warning: {message}')
            if not (mountpoint / 'MyDrive').exists():
                drive.mount(str(mountpoint), force_remount=True)
        else:
            raise
    except Exception as exc:
        print(f'Colab Drive mount skipped or failed: {exc}')


def _is_repo_dir(path: Path) -> bool:
    return (
        (path / 'src' / 'sport_pipeline' / '__init__.py').exists()
        and (path / 'configs').exists()
        and (path / 'notebooks').exists()
    )


def _resolve_repo_dir() -> Path:
    fixed = Path('/content/drive/MyDrive/codex/batting_codex_handoff')
    candidates: list[Path] = []
    env_root = os.environ.get('BATTING_CODE_ROOT')
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend(
        [
            fixed,
            Path.cwd(),
        ]
    )
    candidates.extend(Path.cwd().parents)

    checked: list[str] = []
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if str(candidate) in checked:
            continue
        checked.append(str(candidate))
        if _is_repo_dir(candidate):
            if candidate != fixed:
                print('WARNING: fixed REPO_DIR ではない場所の repo を使います。')
                print('  fixed:', fixed)
                print('  using:', candidate)
                print('  次回は repo フォルダを fixed path に置くか、BATTING_CODE_ROOT を明示してください。')
            return candidate

    raise FileNotFoundError(
        'sport_pipeline package が見つかりません。Drive には notebook 単体ではなく repo フォルダごと置く必要があります。\n'
        '推奨配置: /content/drive/MyDrive/codex/batting_codex_handoff\n'
        '別の場所に置いた場合は、この notebook の最初の cell より前に次を実行してください。\n'
        '  %env BATTING_CODE_ROOT=/content/drive/MyDrive/batting_codex_handoff\n'
        '確認した候補:\n- ' + '\n- '.join(checked)
    )


_mount_drive_if_needed()

REPO_DIR = _resolve_repo_dir()
BASE_DIR = Path('/content/drive/MyDrive/baseball_vision')
CACHE_DIR = Path('/content/cache/baseball_vision')
RUN_PROFILE_NAME = os.environ.get('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
RUN_PROFILE_PATH = REPO_DIR / 'configs' / 'runs' / RUN_PROFILE_NAME

BASE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)

src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if importlib.util.find_spec('sport_pipeline') is None:
    raise ModuleNotFoundError(
        'sport_pipeline を import できません。repo 配置と src/sport_pipeline/__init__.py を確認してください。\n'
        f'REPO_DIR={REPO_DIR}\n'
        f'src_dir={src_dir}\n'
        f'src_dir_exists={src_dir.exists()}'
    )

print('REPO_DIR =', REPO_DIR)
print('BASE_DIR =', BASE_DIR)
print('CACHE_DIR =', CACHE_DIR)
print('RUN_PROFILE_PATH =', RUN_PROFILE_PATH)
print('src_dir =', src_dir)

from sport_pipeline.pipeline.run_profile import load_run_profile, resolve_statcast_date_range, run_id, stage_settings, threshold

RUN_PROFILE = load_run_profile(RUN_PROFILE_PATH)
START_DATE, END_DATE = resolve_statcast_date_range(RUN_PROFILE)
RUN_ID = run_id(RUN_PROFILE, 'full_run_id')

print('STATCAST_DATE_RANGE =', START_DATE, 'to', END_DATE)
print('RUN_ID =', RUN_ID)


## Shard Plan

In [ ]:
from collections import Counter
import json

from sport_pipeline.video.download_sources import (
    download_video_sources,
    plan_video_downloads,
    summarize_download_manifest,
)

SHARD_COUNT = 3
SHARD_INDEX = 0
SHARD_LABEL = f'{SHARD_INDEX + 1}_of_{SHARD_COUNT}'

DOWNLOAD_SETTINGS = stage_settings(RUN_PROFILE, 'video_download')
CONFIG_BATCH_COUNT = int(DOWNLOAD_SETTINGS.get('batch_count') or SHARD_COUNT)
if CONFIG_BATCH_COUNT != SHARD_COUNT:
    raise RuntimeError(f'This notebook is fixed to 3 shards, but profile batch_count={CONFIG_BATCH_COUNT}. Regenerate shard notebooks or set batch_count=3.')

ENABLE_VIDEO_DOWNLOAD = bool(DOWNLOAD_SETTINGS.get('execute_default', True))
MAX_FILES = DOWNLOAD_SETTINGS.get('max_files')
MAX_BYTES_PER_FILE = DOWNLOAD_SETTINGS.get('max_bytes_per_file')
TIMEOUT_SEC = int(DOWNLOAD_SETTINGS.get('timeout_sec', 900))
ALLOW_HLS = bool(DOWNLOAD_SETTINGS.get('allow_hls', True))
DOWNLOAD_NUM_WORKERS = int(DOWNLOAD_SETTINGS.get('num_workers') or 2)
REQUIRE_EVENT_LEVEL_MATCH = bool(DOWNLOAD_SETTINGS.get('require_event_level_match', True))
MIN_DOWNLOAD_MATCH_CONFIDENCE = float(DOWNLOAD_SETTINGS.get('min_match_confidence', 0.45))
PREFER_EXACT_PLAY = bool(DOWNLOAD_SETTINGS.get('prefer_exact_play', True))
MAX_SOURCES_PER_EVENT = DOWNLOAD_SETTINGS.get('max_sources_per_event')
DOWNLOAD_LOG_EVERY = int(DOWNLOAD_SETTINGS.get('log_every') or 1)
DOWNLOAD_MANIFEST_WRITE_EVERY = int(DOWNLOAD_SETTINGS.get('manifest_write_every') or 1)
VIDEO_REUSE_SETTINGS = stage_settings(RUN_PROFILE, 'video_reuse')
REUSE_PREVIOUS_DOWNLOADS = bool(VIDEO_REUSE_SETTINGS.get('reuse_previous_downloads', False))
REUSE_SOURCE_RUN_IDS = [str(item) for item in VIDEO_REUSE_SETTINGS.get('source_full_run_ids', [])]

video_sources = BASE_DIR / 'manifests' / 'video_sources_v1.parquet'
download_dir = BASE_DIR / 'raw_videos' / RUN_ID
shard_manifest = download_dir / f'download_manifest_shard_{SHARD_LABEL}_v1.parquet'
canonical_manifest = download_dir / 'download_manifest_v1.parquet'
reuse_manifest_paths = []
if REUSE_PREVIOUS_DOWNLOADS:
    reuse_manifest_paths = [
        BASE_DIR / 'raw_videos' / source_run_id / 'download_manifest_v1.parquet'
        for source_run_id in REUSE_SOURCE_RUN_IDS
    ]
    if canonical_manifest.exists():
        reuse_manifest_paths.append(canonical_manifest)
shard_progress = BASE_DIR / 'reports/preflight' / f'video_download_{RUN_ID}_shard_{SHARD_LABEL}_progress_v1.json'

if not video_sources.exists():
    raise FileNotFoundError(f'video_sources_v1 がありません。先に notebooks/11_download_statcast_and_video_sources.ipynb の StatsAPI resolve まで実行してください: {video_sources}')

all_plan_rows = plan_video_downloads(
    video_sources_path=video_sources,
    output_dir=download_dir,
    max_files=MAX_FILES,
    batch_count=None,
    include_skipped=False,
    require_event_level_match=REQUIRE_EVENT_LEVEL_MATCH,
    min_match_confidence=MIN_DOWNLOAD_MATCH_CONFIDENCE,
    prefer_exact_play=PREFER_EXACT_PLAY,
    max_sources_per_event=MAX_SOURCES_PER_EVENT,
)
shard_plan_rows = plan_video_downloads(
    video_sources_path=video_sources,
    output_dir=download_dir,
    max_files=MAX_FILES,
    batch_count=SHARD_COUNT,
    batch_index=SHARD_INDEX,
    include_skipped=True,
    require_event_level_match=REQUIRE_EVENT_LEVEL_MATCH,
    min_match_confidence=MIN_DOWNLOAD_MATCH_CONFIDENCE,
    prefer_exact_play=PREFER_EXACT_PLAY,
    max_sources_per_event=MAX_SOURCES_PER_EVENT,
)
planned_rows = [row for row in shard_plan_rows if row.download_status == 'planned']
status_counts = Counter(row.download_status for row in shard_plan_rows)
skip_counts = Counter(row.skip_reason for row in shard_plan_rows if row.skip_reason)

print('SHARD =', SHARD_INDEX + 1, '/', SHARD_COUNT)
print('video_sources ->', video_sources)
print('download_dir ->', download_dir)
print('shard_manifest ->', shard_manifest)
print('canonical_manifest after merge ->', canonical_manifest)
print('shard_progress ->', shard_progress)
print('execute =', ENABLE_VIDEO_DOWNLOAD)
print('GLOBAL max_files =', MAX_FILES, '(split across all shards, not per shard)')
print('num_workers =', DOWNLOAD_NUM_WORKERS, 'max_bytes_per_file =', MAX_BYTES_PER_FILE)
print('timeout_sec =', TIMEOUT_SEC, 'allow_hls =', ALLOW_HLS)
print('require_event_level_match =', REQUIRE_EVENT_LEVEL_MATCH, 'min_match_confidence =', MIN_DOWNLOAD_MATCH_CONFIDENCE)
print('prefer_exact_play =', PREFER_EXACT_PLAY, 'max_sources_per_event =', MAX_SOURCES_PER_EVENT)
print('log_every =', DOWNLOAD_LOG_EVERY, 'manifest_write_every =', DOWNLOAD_MANIFEST_WRITE_EVERY)
print('reuse_previous_downloads =', REUSE_PREVIOUS_DOWNLOADS, 'reuse_source_run_ids =', REUSE_SOURCE_RUN_IDS)
print('reuse_manifest_paths =', [str(path) for path in reuse_manifest_paths])
print('all_planned_downloads_total =', len(all_plan_rows), '(global total across 11_a/11_b/11_c)')
print('shard_planned_downloads =', len(planned_rows), '(this notebook only)')
print('shard_status_counts =', dict(status_counts))
print('shard_skip_reason_counts_top20 =', dict(skip_counts.most_common(20)))
print('existing_shard_manifest_summary =', summarize_download_manifest(shard_manifest))
if shard_progress.exists():
    progress_payload = json.loads(shard_progress.read_text(encoding='utf-8'))
    progress_keys = [
        'status', 'overall_status', 'batch_index', 'batch_count', 'num_workers',
        'total_planned_this_call', 'processed_this_call', 'remaining_this_call',
        'overall_total_planned', 'overall_completed', 'overall_remaining',
        'overall_percent', 'overall_progress_bar', 'overall_status_counts',
        'manifest_rows', 'status_counts', 'elapsed_sec', 'last_video_source_id',
        'last_status', 'last_error',
    ]
    print('latest_shard_progress =', {key: progress_payload.get(key) for key in progress_keys})
else:
    print('latest_shard_progress = none yet')

print('first_planned_rows:')
for row in planned_rows[:30]:
    print(row.video_source_id, row.planned_path)

## Run This Shard

In [ ]:
rows = download_video_sources(
    video_sources_path=video_sources,
    output_dir=download_dir,
    output_manifest=shard_manifest,
    reuse_manifest_paths=reuse_manifest_paths,
    execute=ENABLE_VIDEO_DOWNLOAD,
    max_files=MAX_FILES,
    max_bytes_per_file=MAX_BYTES_PER_FILE,
    timeout_sec=TIMEOUT_SEC,
    allow_hls=ALLOW_HLS,
    batch_count=SHARD_COUNT,
    batch_index=SHARD_INDEX,
    progress_path=shard_progress,
    log_every=DOWNLOAD_LOG_EVERY,
    manifest_write_every=DOWNLOAD_MANIFEST_WRITE_EVERY,
    num_workers=DOWNLOAD_NUM_WORKERS,
    require_event_level_match=REQUIRE_EVENT_LEVEL_MATCH,
    min_match_confidence=MIN_DOWNLOAD_MATCH_CONFIDENCE,
    prefer_exact_play=PREFER_EXACT_PLAY,
    max_sources_per_event=MAX_SOURCES_PER_EVENT,
)
print('finished shard =', SHARD_INDEX + 1, '/', SHARD_COUNT)
print('rows_returned =', len(rows))
print('shard_manifest_summary =', summarize_download_manifest(shard_manifest))
print('NEXT after all 11_a/11_b/11_c complete: notebooks/11_d_merge_video_download_shards.ipynb')